# Clase 9 — CNN 3D e imagen médica (parte 1: los datos)

**Objetivos de hoy**
- Explicar por qué una tomografía necesita una CNN en tres dimensiones.
- Visualizar cortes de un volumen 3D.
- Redimensionar un volumen para prepararlo para el entrenamiento.

Consulta el [capítulo 8 del manual](../manual/08_cnn_3d_imagen_medica.md) para la explicación completa.

**Nota**: el dataset real de la competición de Kaggle pesa varias decenas de gigabytes y requiere credenciales de descarga. Para practicar el pipeline completo sin esa dependencia, generamos aquí volúmenes sintéticos con las mismas dimensiones que un caso real; la lógica de redimensionado y preprocesamiento es idéntica.

## 1. Genera un volumen sintético que simula una tomografía

Cada "paciente" es una pila de cortes 2D; aquí generamos ruido con una forma esférica más brillante en el centro, para simular un nódulo.

In [ ]:
import numpy as np
import tensorflow as tf

def generar_volumen_sintetico(num_cortes=40, alto=128, ancho=128, con_nodulo=True, semilla=None):
    rng = np.random.default_rng(semilla)
    volumen = rng.normal(loc=0.2, scale=0.05, size=(num_cortes, alto, ancho)).astype("float32")
    if con_nodulo:
        centro = (num_cortes // 2, alto // 2, ancho // 2)
        radio = 8
        z, y, x = np.ogrid[:num_cortes, :alto, :ancho]
        mascara_esfera = (z - centro[0]) ** 2 + (y - centro[1]) ** 2 + (x - centro[2]) ** 2 <= radio ** 2
        volumen[mascara_esfera] += 0.6
    return np.clip(volumen, 0, 1)

volumen_ejemplo = generar_volumen_sintetico(con_nodulo=True, semilla=1)
print("Forma del volumen:", volumen_ejemplo.shape)

## 2. Visualiza varios cortes del volumen

Igual que un radiólogo revisa corte a corte, aquí proyectamos algunos cortes de la pila.

In [ ]:
import matplotlib.pyplot as plt

cortes_a_mostrar = [10, 15, 20, 25, 30, 35]
plt.figure(figsize=(12, 4))
for i, indice_corte in enumerate(cortes_a_mostrar):
    plt.subplot(1, len(cortes_a_mostrar), i + 1)
    plt.imshow(volumen_ejemplo[indice_corte], cmap="gray")
    plt.title(f"Corte {indice_corte}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 3. Redimensiona el volumen: alto, ancho y también profundidad

`tf.image.resize` solo trabaja sobre dos dimensiones espaciales a la vez, así que transponemos el volumen para redimensionar primero alto/ancho y después la profundidad.

In [ ]:
def redimensionar_volumen(volumen, tamano_destino=(64, 64, 32)):
    volumen = tf.expand_dims(volumen, axis=-1)
    volumen = tf.image.resize(volumen, tamano_destino[:2])
    volumen = tf.transpose(volumen, perm=[2, 0, 1, 3])
    volumen = tf.image.resize(volumen, [tamano_destino[2], tamano_destino[0]])
    volumen = tf.transpose(volumen, perm=[1, 2, 0, 3])
    return volumen

volumen_redimensionado = redimensionar_volumen(volumen_ejemplo)
print("Forma tras redimensionar:", volumen_redimensionado.shape)

## 4. Genera un pequeño dataset de "pacientes" sanos y con nódulo

In [ ]:
def construir_dataset_sintetico(num_pacientes=60, tamano_destino=(64, 64, 32)):
    volumenes = []
    etiquetas = []
    for i in range(num_pacientes):
        con_nodulo = i % 2 == 0
        volumen = generar_volumen_sintetico(con_nodulo=con_nodulo, semilla=i)
        volumen = redimensionar_volumen(volumen, tamano_destino)
        volumenes.append(volumen.numpy())
        etiquetas.append(1 if con_nodulo else 0)
    return np.array(volumenes), np.array(etiquetas)

X, y = construir_dataset_sintetico()
print("Forma del dataset de volúmenes:", X.shape)
print("Distribución de etiquetas:", np.bincount(y))

np.save("X_cnn3d.npy", X)
np.save("y_cnn3d.npy", y)

## Resumen

- Un volumen 3D es una pila de cortes 2D; para redimensionarlo hay que tratar también la dimensión de profundidad.
- Guardaste `X_cnn3d.npy` y `y_cnn3d.npy` con el dataset ya preprocesado, listos para entrenar en la próxima clase.

Continúa con el [notebook de práctica de esta clase](../notebooks_practica/clase09_practica_cnn3d_datos.ipynb).